# DAY - 29
# DATE - 19.06.2026
# EuroSAT  Training Pipeline

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EuroSAT
import torch.nn as nn
import time

In [2]:
# 1. Transforms & Augmentation
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],
                         std = [0.229, 0.224, 0.225])
])

# 2. Download EuroSAT (Will download to ./data)
dataset = EuroSAT(root="./data", download=True, transform=transform)

100%|██████████| 94.3M/94.3M [00:00<00:00, 352MB/s]


In [3]:
# Split into Train (80%) and Val (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size = 64, shuffle = False)

# 3. Model setup (ResNet18)
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

# EuroSAT has 10 classes
model.fc = nn.Linear(model.fc.in_features, 10)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models = model.to(device)

criterion = nn.CrossEntropyLoss()
# Small learning rate for transfer learning to avoid catastrophic forgetting
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


In [5]:
# 4. Training Loop
start_time = time.time()
print("Starting Training...")

for epoch in range(5):  # 5 epochs are usually enough to cross 90% with TL
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}: Validation Accuracy = {accuracy:.2f}%")

print(f"Total Training Time: {time.time() - start_time:.2f} seconds")

Starting Training...
Epoch 1: Validation Accuracy = 96.07%
Epoch 2: Validation Accuracy = 96.54%
Epoch 3: Validation Accuracy = 96.65%
Epoch 4: Validation Accuracy = 96.39%
Epoch 5: Validation Accuracy = 97.04%
Total Training Time: 109.94 seconds
